In [62]:
import torch
import json
import os

mode='rewrite'
pred_root=f"exp_{mode}/shikra_eval_rec_human/custom_metrics.pth"
json_path='/home/v-jinjzhao/datasets/rec_human/person_annts_val_ready.json'
preds = torch.load(pred_root)
with open(json_path) as f:
    all_annts=json.load(f)

print(preds.keys())
print(f"normalized acc: {len(preds['pred_boxes'])}")

dict_keys(['accuracy', 'target_failed', 'failed', 'iou', 'warning', 'pred_boxes'])
normalized acc: 20609


In [63]:
preds['accuracy']

0.3726042020476491

## following are custom eval code 
## save json use this

*input:*
-   gt_bboxes: read width and height, absolute [xyxy]
-   pred_bboxes: read width and height, absolute [xyxy]

In [64]:
import cv2
import matplotlib.pyplot as plt

def box_xyxy_expand2square(box, *, w, h):
    if w == h:
        return box
    if w > h:
        x1, y1, x2, y2 = box
        y1 += (w - h) // 2
        y2 += (w - h) // 2
        box = x1, y1, x2, y2
        return box
    assert w < h
    x1, y1, x2, y2 = box
    x1 += (h - w) // 2
    x2 += (h - w) // 2
    box = x1, y1, x2, y2
    return box
# write bbox_xyxy_de-square
def box_xyxy_de_expand2square(box, *, w, h):
    if w == h:
        return box
    if w > h:
        x1, y1, x2, y2 = box
        y1 -= (w - h) // 2
        y2 -= (w - h) // 2
        box = x1, y1, x2, y2
        return box
    assert w < h
    x1, y1, x2, y2 = box
    x1 -= (h - w) // 2
    x2 -= (h - w) // 2
    box = x1, y1, x2, y2
    return box

img_folder='/home/v-jinjzhao/datasets/jierun/images'
pred_bboxes=[]
gt_bboxes=[]
for idx,annt in enumerate(all_annts):
    pred_bbox=preds['pred_boxes'][idx]
    # print(pred_bbox)
    img_path=annt['file_name']
    height,width=annt['height'],annt['width']

    gt_bbox=annt['bbox']
    gt_bbox=[gt_bbox[0],gt_bbox[1],gt_bbox[0]+gt_bbox[2],gt_bbox[1]+gt_bbox[3]]
    if(pred_bbox!=[0,0,0,0]):
        pred_bbox=list(box_xyxy_de_expand2square(pred_bbox*max(width,height), w=width, h=height))
    gt_bboxes.append(gt_bbox)
    pred_bboxes.append(pred_bbox)
    
    # draw bbox
    # img=cv2.imread(os.path.join(img_folder,img_path))
    # img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    # img=cv2.rectangle(img,(gt_bbox[0],gt_bbox[1]),(gt_bbox[2],gt_bbox[3]),(0,255,0),2)
    # img=cv2.rectangle(img,(pred_bbox[0],pred_bbox[1]),(pred_bbox[2],pred_bbox[3]),(255,0,0),2)
    # plt.imshow(img)
    # plt.show()


In [65]:
import json
import torch
from tqdm import tqdm
from overlaps import bbox_overlaps
# import average from math
from statistics import mean


def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs




pred_bboxes=torch.tensor(pred_bboxes)
gt_bboxes=torch.tensor(gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.5220049492939978
thresh=0.7, acc=0.3619777766995002
thresh=0.9, acc=0.08292493570770051
iou|0.5:0.9=0.33720434977167474
tensor([0.3882, 0.3721, 0.0000,  ..., 0.6098, 0.5542, 0.0000])
{0.5: 0.5220049492939978, 0.55: 0.4874569362899704, 0.6: 0.4498034839147945, 0.65: 0.40802561987481195, 0.7: 0.3619777766995002, 0.75: 0.3088941724489301, 0.8: 0.24518414285021106, 0.85: 0.168567130865156, 0.9: 0.08292493570770051}
20609/20609


In [61]:
for iou_i, pred_bbox, annt in zip(iou,pred_bboxes,all_annts):
    annt[f'shikra7b_{mode}_pred']=dict(
        iou=iou_i.item(),
        pred_bbox=pred_bbox.tolist()
    )
save=input("press 'yes' to save")
if(save=='yes'):
    with open(json_path,'w') as f:
        # save and format
        json.dump(all_annts,f,indent=4)
    print(f"save to {json_path}")
else:
    print("not save")

save to /home/v-jinjzhao/datasets/rec_human/person_annts_val_ready.json


In [51]:
with open(json_path) as f:
    all_annts=json.load(f)
for annt in all_annts:
    if('shikra7b_caption_pred' in annt):
        annt.pop('shikra7b_caption_pred')
    if('shikra7b_rewrite_pred' in annt):
        annt.pop('shikra7b_rewrite_pred')
    if('shikra_rewrite_pred' in annt):
        annt.pop('shikra_rewrite_pred')
    if('shikra_caption_pred' in annt):
        annt.pop('shikra_caption_pred')
with open(json_path,'w') as f:
    json.dump(all_annts,f,indent=4)

## following are raw eval code 

*input:*
-   gt_bboxes: squared, normalized [xyxy]
-   pred_bboxes: squared, normalized [xyxy]

In [25]:
import cv2
import matplotlib.pyplot as plt

def box_xyxy_expand2square(box, *, w, h):
    if w == h:
        return box
    if w > h:
        x1, y1, x2, y2 = box
        y1 += (w - h) // 2
        y2 += (w - h) // 2
        box = x1, y1, x2, y2
        return box
    assert w < h
    x1, y1, x2, y2 = box
    x1 += (h - w) // 2
    x2 += (h - w) // 2
    box = x1, y1, x2, y2
    return box
# write bbox_xyxy_de-square
def box_xyxy_de_expand2square(box, *, w, h):
    if w == h:
        return box
    if w > h:
        x1, y1, x2, y2 = box
        y1 -= (w - h) // 2
        y2 -= (w - h) // 2
        box = x1, y1, x2, y2
        return box
    assert w < h
    x1, y1, x2, y2 = box
    x1 -= (h - w) // 2
    x2 -= (h - w) // 2
    box = x1, y1, x2, y2
    return box

img_folder='/home/v-jinjzhao/datasets/jierun/images'
pred_bboxes=[]
gt_bboxes=[]
for idx,annt in enumerate(all_annts):
    pred_bbox=preds['test_pred_boxes'][idx]
    # print(pred_bbox)
    img_path=annt['file_name']
    height,width=annt['height'],annt['width']

    gt_bbox=annt['bbox']
    gt_bbox=[gt_bbox[0],gt_bbox[1],gt_bbox[0]+gt_bbox[2],gt_bbox[1]+gt_bbox[3]]
    gt_bbox=box_xyxy_expand2square(gt_bbox, w=width, h=height)
    gt_bbox=[x/max(width,height) for x in gt_bbox]
    gt_bboxes.append(gt_bbox)
    pred_bboxes.append(pred_bbox.tolist())
    
    # draw bbox
    # img=cv2.imread(os.path.join(img_folder,img_path))
    # img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    # img=cv2.rectangle(img,(gt_bbox[0],gt_bbox[1]),(gt_bbox[2],gt_bbox[3]),(0,255,0),2)
    # img=cv2.rectangle(img,(pred_bbox[0],pred_bbox[1]),(pred_bbox[2],pred_bbox[3]),(255,0,0),2)
    # plt.imshow(img)
    # plt.show()


In [26]:

import json
import torch
from tqdm import tqdm
from torchvision.ops.boxes import box_iou
# import average from math
from statistics import mean

def calculate_iou_acc_mean(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    ious=box_iou(pred_bboxes*1000,gt_bboxes*1000)
    ious = torch.einsum('i i -> i', ious)  # take diag elem
    # NOTE: please note iou only calculate for success target
    iou = ious.mean().item()
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        correct = (ious > t).sum().item()
        accs[t]=correct/len(ious)
    return ious,accs,iou

pred_bboxes=torch.tensor(pred_bboxes)
gt_bboxes=torch.tensor(gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc,mean_iou=calculate_iou_acc_mean(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]


thresh=0.5, acc=0.6505149864361174
thresh=0.7, acc=0.46253942347985266
thresh=0.9, acc=0.10412209699830176
iou|0.5:0.9=0.42632986088137054
tensor([0.8363, 0.8541, 0.9312,  ..., 0.7110, 0.7818, 0.8018])
{0.5: 0.6505149864361174, 0.55: 0.6105291016960367, 0.6: 0.5667938510398977, 0.65: 0.5195077303103152, 0.7: 0.46253942347985266, 0.75: 0.39606537129750113, 0.8: 0.3147923512935312, 0.85: 0.2121038353807812, 0.9: 0.10412209699830176}
45341/45341


In [72]:
for iou_i, pred_bbox, annt in zip(iou, pred_bboxes, all_annts):
    annt[f'shikra_{mode}_pred']=dict(
        iou=iou_i.item(),
        pred_bbox=pred_bbox.tolist()
    )
with open(json_path,'w') as f:
    # save and format
    json.dump(all_annts,f,indent=4)

In [6]:
import torch
x=torch.load('exp_caption/shikra_eval_rec_human/sample_check_test_-1_1.pt')


In [1]:
from mllm.config import prepare_args
from mllm.models import load_pretrained
from mllm.utils import print_trainable_params
from mllm.engine import prepare_trainer_collator
from mllm.dataset import prepare_data, prepare_target_processor

cfg, training_args = prepare_args()
model, preprocessor = load_pretrained(cfg.model_args, training_args)
# Some ugly codes to inject target_processor into preprocessor.
# maybe effect model. (e.g. add special token; resize embedding)
model, preprocessor = prepare_target_processor(model, preprocessor, cfg.model_args, training_args)
print_trainable_params(model)

# Prepare data_collator
collator_kwargs = cfg.data_args.collator_kwargs
trainer_cls, data_collator_dict = prepare_trainer_collator(cfg.model_args, preprocessor, collator_kwargs)
dataset, compute_metrics = prepare_data(cfg.data_args, cfg.model_args, training_args, preprocessor)

/home/v-jinjzhao/miniconda3/envs/shikra/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
usage: ipykernel_launcher.py [-h] [--output_dir OUTPUT_DIR]
                             [--overwrite_output_dir [OVERWRITE_OUTPUT_DIR]]
                             [--do_train [DO_TRAIN]] [--do_eval [DO_EVAL]]
                             [--do_predict [DO_PREDICT]]
                             [--evaluation_strategy {no,steps,epoch}]
                             [--prediction_loss_only [PREDICTION_LOSS_ONLY]]
                             [--per_device_train_batch_size PER_DEVICE_TRAIN_BATCH_SIZE]
                             [--per_device_eval_batch_size PER_DEVICE_EVAL_BATCH_SIZE]
                             [--per_gpu_train_batch_size PER_GPU_TRAIN_BATCH_SIZE]
                             [--per_gpu_eval_batch_

SystemExit: 2

/home/v-jinjzhao/miniconda3/envs/shikra/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
